# Lane Qwen2-VL-7B-Instruct (Zero-Shot) — PC Kampus RTX 3060

Versi 7B, khusus untuk dijalankan di **PC kampus (RTX 3060, 12,9 GB VRAM)**. Model visual-language dipakai **langsung sebagai classifier**, bukan untuk melabeli Data Uji. Tidak ada fine-tuning — murni prompting/inferensi ke bobot pre-trained publik.

## Kenapa 7B, dan kenapa harus di kampus

- 7B parameter jauh lebih akurat untuk task visual reasoning dibanding 2B, tapi butuh VRAM jauh lebih besar.
- Di BF16 penuh (tanpa quantization): ~15–16 GB VRAM untuk bobot saja — **tidak muat** di RTX 3050 (4,3 GB), bahkan kemungkinan mepet di RTX 3060 (12,9 GB).
- Karena itu notebook ini **default ke 4-bit quantization** (~6 GB) supaya aman di RTX 3060 dengan sisa VRAM untuk vision encoder + KV-cache. BF16 penuh disediakan sebagai opsi kalau kamu sudah tes dan RTX 3060 sanggup (butuh cek langsung, bukan diasumsikan).
- Unduhan model **~15 GB** (7B x 2 byte BF16) — jauh lebih besar dari versi 2B (~4-5GB). Pastikan koneksi kampus dan ruang disk cukup sebelum mulai.

## Kenapa ini aman menurut juknis

- Model pre-trained publik, tidak pernah dilatih di Data Latih/Data Uji kompetisi ini (pasal 5).
- Data Uji hanya dipakai untuk **menghasilkan prediksi akhir** — tidak ada label Data Uji yang masuk ke proses apa pun di notebook ini.
- Zero-shot berarti tidak ada `.fit()`/training; hanya `.generate()` untuk inferensi.
- Tidak ada sel yang membaca `solution.csv` atau berkas revisi manual apa pun — disengaja.

## Realita waktu

- 7B lebih lambat per gambar dibanding 2B (lebih banyak parameter untuk di-forward).
- Estimasi 1.458 gambar Data Uji: siapkan **2–4 jam** tergantung `max_new_tokens` dan beban lain di GPU kampus.
- **Resume-safe**: hasil disimpan progresif ke JSON, aman dihentikan dan dilanjutkan — penting karena sesi PC kampus mungkin tidak bisa dibiarkan menyala semalaman tanpa pengawasan.

In [ ]:
# =============================== KONFIGURASI ===============================
import os, gc, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch

CANDIDATE_ROOTS = [
    r"C:\Users\MyPC PRO\Downloads\BDC2026",   # PC kampus -- prioritas untuk notebook ini
    r"D:\Downloads\BDC2026",
]
ROOT = next((Path(p) for p in CANDIDATE_ROOTS if Path(p).exists()), Path.cwd())
DATA_DIR = ROOT / "clean_dataset_v3"   # dataset asli ada di sini, bukan langsung di ROOT

TRAIN_DIR    = DATA_DIR / "train"
TEST_DIR     = DATA_DIR / "test"
TEMPLATE_CSV = DATA_DIR / "submission.csv"     # template resmi panitia -- untuk urutan id saja

OUT = ROOT / "stack_out"
QWEN_DIR = OUT / "qwen7b_lane"
QWEN_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["0_Recyclable", "1_Electronic", "2_Organic"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / 1024**3
    print(f"GPU  : {props.name}  ({VRAM_GB:.1f} GB)")
else:
    VRAM_GB = 0.0
    print("GPU  : tidak terdeteksi -- JANGAN lanjut, 7B di CPU akan sangat sangat lambat")

# --- pilihan precision ---
# "4bit"  : ~6 GB, paling aman untuk RTX 3060, akurasi sedikit di bawah BF16
# "bf16"  : ~15-16 GB, presisi penuh, HANYA coba kalau VRAM >= 16GB atau sudah tes muat
PRECISION = "4bit" if VRAM_GB < 16 else "bf16"

print(f"ROOT      : {ROOT}")
print(f"Precision : {PRECISION}")
if PRECISION == "4bit" and VRAM_GB >= 16:
    print("(VRAM cukup untuk BF16, tapi tetap default ke 4bit -- ganti manual di sini kalau mau presisi penuh)")
if VRAM_GB < 10:
    print("\n!! PERINGATAN: VRAM di bawah 10GB. Bahkan versi 4-bit 7B mungkin OOM.")
    print("   Kalau OOM terjadi, turunkan MAX_PIXELS di sel berikutnya, atau pakai versi 2B saja.")

In [ ]:
# =============================== INSTALASI (jalankan sekali) ===============================
# pip install -U transformers accelerate qwen-vl-utils bitsandbytes pillow
#
# Kalau bitsandbytes gagal di Windows: set PRECISION = "bf16" manual di sel sebelumnya
# dan pastikan VRAM benar-benar cukup (>=16GB), atau turun ke versi 2B.
print("Cek dependensi lewat: pip show transformers accelerate qwen-vl-utils bitsandbytes")

## Muat model

Unduhan ~15 GB (bobot BF16 penuh diunduh dulu, baru dikuantisasi saat load kalau `PRECISION="4bit"`). Sekali unduh, di-cache lokal -- sesi berikutnya tidak mengunduh ulang.

In [ ]:
# =============================== MUAT QWEN2-VL-7B-INSTRUCT ===============================
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

# Guard: kalau model sudah ke-load di sesi kernel ini (mis. gara-gara "Run All" nge-trigger
# cell ini lagi), JANGAN load ulang -- instance kedua akan berebut VRAM sama yang pertama
# dan gagal dengan ValueError "dispatched on the CPU or the disk".
if "model" in dir() and "processor" in dir():
    print("Model & processor sudah ke-load sebelumnya di sesi ini -- skip, pakai yang ada.")
else:
    t0 = time.time()
    if PRECISION == "4bit":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID, quantization_config=bnb_config, device_map="auto",
        )
    else:
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto",
        )
    model.eval()

    # Batasi resolusi dinamis Qwen2-VL supaya VRAM terkendali. 7B sudah lebih berat dari 2B,
    # jadi MAX_PIXELS di sini sedikit lebih konservatif dibanding versi 2B.
    MIN_PIXELS = 256 * 28 * 28
    MAX_PIXELS = 512 * 28 * 28   # turunkan ke 384*28*28 kalau masih OOM
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)

    print(f"Model dimuat dalam {(time.time()-t0)/60:.1f} menit")
    if DEVICE == "cuda":
        print(f"VRAM terpakai: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
        print(f"VRAM total   : {VRAM_GB:.1f} GB")

## Prompt classifier

Sama seperti versi 2B: prompt memakai definisi persis dari juknis (bukan tebakan bebas) supaya konsisten dengan batas kelas yang dipakai panitia, dan keluarannya gampang di-parse.

In [ ]:
# =============================== PROMPT & PARSER ===============================
PROMPT = (
    "Kamu adalah sistem klasifikasi sampah. Lihat gambar ini dan klasifikasikan "
    "ke SATU dari tiga kategori berikut:\n\n"
    "RECYCLABLE: sampah non-elektronik yang bisa didaur ulang "
    "(botol plastik, kaleng, kertas, kardus, kaca)\n"
    "ELECTRONIC: perangkat elektronik atau limbah elektronik, berfungsi maupun rusak "
    "(HP, laptop, keyboard, mouse, charger, kabel)\n"
    "ORGANIC: bahan hayati yang mudah terurai "
    "(daun, buah, sayuran, sisa makanan, ranting)\n\n"
    "Jawab HANYA dengan satu kata: RECYCLABLE, ELECTRONIC, atau ORGANIC. "
    "Tidak ada penjelasan lain."
)

LABEL2IDX = {"RECYCLABLE": 0, "ELECTRONIC": 1, "ORGANIC": 2}

def parse_label(text):
    t = text.upper()
    for name, idx in LABEL2IDX.items():
        if name in t:
            return idx
    return None    # gagal parse -> ditandai, jangan diam-diam ditebak

print("Prompt siap. Contoh kategori & indeks:", LABEL2IDX)

In [ ]:
# =============================== FUNGSI INFERENSI SATU GAMBAR ===============================
from PIL import Image
from qwen_vl_utils import process_vision_info

@torch.no_grad()
def classify_image(path, max_new_tokens=10):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(path)},
            {"type": "text", "text": PROMPT},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)

    out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    raw = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    return raw, parse_label(raw)

# uji cepat di 1 gambar sebelum jalan penuh -- cek juga VRAM peak di sini
_test_img = next(iter((TRAIN_DIR / CLASSES[0]).glob("*")))
_t0 = time.time()
raw, idx = classify_image(_test_img)
print(f"Contoh   : {_test_img.name}")
print(f"Keluaran : '{raw}'  ->  indeks {idx} ({CLASSES[idx] if idx is not None else 'GAGAL PARSE'})")
print(f"Waktu    : {time.time()-_t0:.1f} detik/gambar (estimasi kasar untuk ETA)")
if DEVICE == "cuda":
    print(f"VRAM peak: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

## Jalankan ke seluruh Data Uji

Progresif dan resume-safe: hasil disimpan tiap `SAVE_EVERY` gambar ke JSON. Penting untuk 7B karena waktu totalnya lebih panjang dari versi 2B -- kalau sesi PC kampus perlu dihentikan di tengah jalan (misal lab tutup), tinggal lanjutkan nanti tanpa mengulang dari nol.

In [ ]:
# =============================== INDEKS DATA UJI (urutan dari template panitia) ===============================
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"}
test_files = [p for p in TEST_DIR.iterdir() if p.suffix.lower() in IMG_EXT]
stem2path = {p.stem: p for p in test_files}
num2path = {}
for p in test_files:
    digits = "".join(ch for ch in p.stem if ch.isdigit())
    if digits:
        num2path.setdefault(int(digits), p)

assert TEMPLATE_CSV.exists(), f"Template panitia tidak ada di {TEMPLATE_CSV}"
TEMPLATE = pd.read_csv(TEMPLATE_CSV)
ID_COL = TEMPLATE.columns[0]

def resolve(raw):
    s = str(raw).strip()
    for cand in (s, Path(s).stem):
        if cand in stem2path:
            return stem2path[cand]
    digits = "".join(ch for ch in s if ch.isdigit())
    if digits and int(digits) in num2path:
        return num2path[int(digits)]
    return None

resolved = [resolve(v) for v in TEMPLATE[ID_COL]]
missing = [v for v, r in zip(TEMPLATE[ID_COL], resolved) if r is None]
assert not missing, f"{len(missing)} id template tidak ketemu filenya"

test_df = pd.DataFrame({"id": TEMPLATE[ID_COL].values, "path": [str(p) for p in resolved]})
print(f"Data Uji: {len(test_df)} gambar (juknis: 1.458), urutan mengikuti template panitia")

In [ ]:
# =============================== INFERENSI PROGRESIF (RESUME-SAFE) ===============================
RESULT_PATH = QWEN_DIR / "qwen7b_test_predictions.json"
SAVE_EVERY = 20    # lebih sering dari versi 2B karena tiap gambar lebih mahal waktu

results = {}
if RESULT_PATH.exists():
    results = json.loads(RESULT_PATH.read_text())
    print(f"Melanjutkan dari {len(results)} hasil yang sudah ada")

todo = [(i, r) for i, r in test_df.iterrows() if r["id"] not in results]
print(f"Sisa yang perlu diproses: {len(todo)} / {len(test_df)}")

t0 = time.time()
for n, (i, row) in enumerate(todo):
    raw, idx = classify_image(row["path"])
    results[str(row["id"])] = {"raw": raw, "label_idx": idx}

    if (n + 1) % SAVE_EVERY == 0 or (n + 1) == len(todo):
        RESULT_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=1))
        elapsed = time.time() - t0
        rate = (n + 1) / elapsed
        eta_min = (len(todo) - (n + 1)) / rate / 60 if rate > 0 else float("inf")
        print(f"  [{n+1}/{len(todo)}]  {elapsed/60:.1f} menit berlalu  "
              f"ETA sisa: {eta_min:.1f} menit  (tersimpan)")

print(f"\nSelesai. Total hasil tersimpan: {len(results)}")

In [ ]:
# =============================== RANGKUM HASIL -> LANE PROBABILITAS ===============================
# Qwen dipakai sebagai classifier hard-label (bukan softmax), jadi "probabilitas" di sini
# berbentuk one-hot -- tetap kompatibel untuk digabung ke ensemble sebagai satu lane.

failed = [k for k, v in results.items() if v["label_idx"] is None]
if failed:
    print(f"!! {len(failed)} gambar gagal di-parse (keluaran di luar 3 kategori). Contoh:")
    for k in failed[:5]:
        print(f"   id={k}  keluaran='{results[k]['raw']}'")
    print("Ditandai sebagai NaN dulu -- putuskan manual: retry dengan prompt lebih ketat, atau fallback ke lane lain.")

qwen_pred = np.full(len(test_df), -1, dtype=np.int64)
for i, row in test_df.iterrows():
    idx = results.get(str(row["id"]), {}).get("label_idx")
    if idx is not None:
        qwen_pred[i] = idx

qwen_onehot = np.zeros((len(test_df), 3), dtype=np.float32)
valid = qwen_pred >= 0
qwen_onehot[valid, qwen_pred[valid]] = 1.0

np.save(QWEN_DIR / "qwen7b_test.npy", qwen_onehot)
print(f"\nDisimpan -> {QWEN_DIR / 'qwen7b_test.npy'}")
print(f"Valid    : {valid.sum()} / {len(test_df)}")
print("\nSebaran prediksi Qwen2-VL-7B (data uji):")
for c, n in zip(CLASSES, np.bincount(qwen_pred[valid], minlength=3)):
    print(f"  {c:<16} {n:>5}")

## Menggabungkan ke ensemble

`qwen7b_test.npy` berbentuk sama seperti lane probabilitas lain (`N_test x 3`), tinggal ditambahkan ke daftar `probs_list` di notebook ensemble kamu. Kalau file ini dihasilkan di PC kampus, pindahkan ke laptop lewat folder `stack_out/qwen7b_lane/` sebelum digabung.

**Tidak ada validasi F1 Qwen di sini dari label Data Uji** -- disengaja. Untuk tahu performa 7B di data berlabel, jalankan sel di bawah ke **Data Latih**, supaya kamu tahu apakah lane ini worth dipakai, sepenuhnya di dalam batas juknis.

In [ ]:
# =============================== (OPSIONAL) VALIDASI DI DATA LATIH ===============================
# Ini AMAN karena Data Latih memang boleh dipakai untuk validasi/model selection.
# Ambil sampel kecil dulu -- 7B lebih lambat dari 2B, jangan langsung ke semua 26.527 gambar.
from sklearn.metrics import f1_score, classification_report
import random

N_SAMPLE = 300     # naikkan kalau waktu masih ada, tapi 7B lebih mahal per gambar dari 2B
random.seed(42)

rows = []
for ci, cname in enumerate(CLASSES):
    folder = TRAIN_DIR / cname
    paths = [p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT]
    for p in random.sample(paths, min(N_SAMPLE // 3, len(paths))):
        rows.append({"path": p, "label": ci})
val_df = pd.DataFrame(rows)
print(f"Sampel validasi (Data Latih): {len(val_df)} gambar")

VAL_RESULT_PATH = QWEN_DIR / "qwen7b_train_sample_predictions.json"
val_results = json.loads(VAL_RESULT_PATH.read_text()) if VAL_RESULT_PATH.exists() else {}

t0 = time.time()
for n, row in val_df.iterrows():
    key = str(row["path"])
    if key in val_results:
        continue
    raw, idx = classify_image(row["path"])
    val_results[key] = {"raw": raw, "label_idx": idx}
    if (n + 1) % 20 == 0:
        VAL_RESULT_PATH.write_text(json.dumps(val_results, ensure_ascii=False, indent=1))

VAL_RESULT_PATH.write_text(json.dumps(val_results, ensure_ascii=False, indent=1))
print(f"Selesai dalam {(time.time()-t0)/60:.1f} menit")

val_df["pred"] = [val_results.get(str(p), {}).get("label_idx") for p in val_df["path"]]
ok = val_df["pred"].notna()
print(f"\nValid parse: {ok.sum()} / {len(val_df)}")
print(f"Macro F1 Qwen2-VL-7B zero-shot (sampel Data Latih): "
      f"{f1_score(val_df.loc[ok, 'label'], val_df.loc[ok, 'pred'], average='macro'):.4f}")
print()
print(classification_report(val_df.loc[ok, "label"], val_df.loc[ok, "pred"],
                            target_names=CLASSES, digits=4))

---
## Ringkasan

| Tahap | Data yang disentuh | Status |
|---|---|---|
| Muat model, prompt | tidak ada | aman |
| Inferensi ke Data Uji | gambar Data Uji saja, tanpa label | aman -- hanya prediksi akhir |
| Validasi F1 | sampel Data Latih (berlabel) | aman -- Data Latih boleh untuk validasi |

Tidak ada sel yang membaca `solution.csv` atau berkas revisi manual apa pun. Bandingkan macro F1 7B ini dengan versi 2B (kalau sudah dicoba) dan dengan SigLIP2 fine-tuned kamu (0,9874) sebelum memutuskan lane mana yang masuk ensemble final -- 7B lebih mahal waktu, jadi worth-nya tergantung seberapa besar kenaikan F1 dibanding biaya komputasinya.

### Kalau OOM di RTX 3060

1. Turunkan `MAX_PIXELS` di sel muat model, misal ke `384*28*28`.
2. Pastikan `PRECISION = "4bit"` (bukan `"bf16"`).
3. Tutup aplikasi lain yang memakai GPU sebelum menjalankan notebook.
4. Kalau masih OOM, turunkan ke versi 2B -- perbedaan akurasi mungkin tidak sebanding dengan risiko gagal jalan sama sekali menjelang deadline.